# Experiment 1: Head-to-head Benchmarks (Med3-TabPFN / Med3-LoCalPFN vs 3D baselines)

This notebook benchmarks the two PFN-based heads against 3D baselines per dataset registered in `configs/datasets.yaml`.

Methods compared:
- Med3-TabPFN (`run_multi_tabpfn`)
- Med3-LoCalPFN (`run_multi_localpfn`)
- DenseNet121-3D (`train_eval_densenet121_3d`)
- ViT-3D (`train_eval_vit_3d`)

Notes:
- PFN heads use a feature-level STRATIFIED split for evaluation (implemented inside the pipelines).


In [1]:
# Environment and import path setup
import os, sys, site, torch
os.environ['PYTHONNOUSERSITE'] = '1'
usr = site.getusersitepackages(); sys.path = [p for p in sys.path if p != usr]

# Reduce thread contention
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'
torch.set_num_threads(1)

from pathlib import Path
def _add_repo_root_to_sys_path():
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        if (base / 'med3pipe').is_dir():
            if str(base) not in sys.path:
                sys.path.insert(0, str(base))
            print('Added repo root to sys.path:', base)
            return base
    raise RuntimeError("Could not locate 'med3pipe/' in current or parent directories.")

repo_root = _add_repo_root_to_sys_path()


Added repo root to sys.path: C:\Users\cahel\Desktop\Med3Tab-PFN


In [2]:
# Configuration
from med3pipe.pipelines import run_multi_tabpfn, run_multi_localpfn
from med3pipe.data.prepare import Sam3DPaths, find_default_sam3d_root
from med3pipe.vision.v3d import train_eval_densenet121_3d, train_eval_vit_3d, Train3DConfig
import yaml
import pandas as pd

config_path = repo_root / 'configs' / 'datasets.yaml'
outputs_base = repo_root / 'notebooks'
dataset_filter = None  # e.g., ['gist', 'lipo']

print('Config path:', config_path)
print('Exists:', config_path.exists())
print('Outputs base:', outputs_base)

with open(config_path, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f) or {}
assert 'datasets' in cfg and isinstance(cfg['datasets'], dict), 'Config must define a datasets: mapping'
all_ds = list(cfg['datasets'].keys())
datasets_to_run = dataset_filter or all_ds
print('Datasets to run:', datasets_to_run)

# Auto-detect SAM-Med3D checkpoint
sam3d_root = find_default_sam3d_root()
checkpoint_path = sam3d_root / 'ckpt' / 'sam_med3d_turbo.pth'
if not checkpoint_path.exists():
    checkpoint_path = sam3d_root / 'ckpt' / 'SAM-Med3D-turbo.pth'
if not checkpoint_path.exists():
    print("⚠️  WARNING: No SAM-Med3D checkpoint found! Model will use random weights.")
    print(f"   Download from: https://huggingface.co/blueyo0/SAM-Med3D/blob/main/sam_med3d_turbo.pth")
    print(f"   Save to: {sam3d_root / 'ckpt' / 'sam_med3d_turbo.pth'}")
    checkpoint_path = None
else:
    print(f"✅ Using SAM-Med3D checkpoint: {checkpoint_path}")


c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Config path: C:\Users\cahel\Desktop\Med3Tab-PFN\configs\datasets.yaml
Exists: True
Outputs base: C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks
Datasets to run: ['gist', 'lipo']
✅ Using SAM-Med3D checkpoint: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\ckpt\sam_med3d_turbo.pth


## Run PFN methods across datasets


In [3]:
# TabPFN
res_tab = run_multi_tabpfn(
    checkpoint=checkpoint_path,
    skip_existing_embeddings=False,
    config_path=config_path,
    dataset_names=datasets_to_run,
    outputs_base_dir=outputs_base,

    # tabpfn_clf_kwargs={'N_ensemble_configurations': 16},
)
res_tab['summary_df']



✅ gist: Reusing existing embeddings...
Prepared 25 cases ...
Prepared 50 cases ...
Prepared 75 cases ...
Prepared 100 cases ...
Prepared 125 cases ...
Prepared 150 cases ...
Prepared 175 cases ...
Prepared 200 cases ...
Prepared 225 cases ...
Done. Prepared 246 cases to C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\train\gist\ct_GIST
Validation set copied to: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\validation\gist\ct_GIST | Train: 196 | Val: 50
Loaded checkpoint (model_state_dict)
To extract: 246 from C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\train\gist\ct_GIST\imagesTr
Done extracting to: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\features\gist\ct_GIST_train
To extract: 50 from C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\validation\gist\ct_GIST\imagesVal
Done extracting to: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\features\gist\ct_G

,dataset,category,ct_name,method,accuracy,macro_f1,roc_auc,out_dir,metrics_path,pred_path,error
0,gist,gist,ct_GIST,tabpfn,0.660000,0.659864,0.744000,C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\t...,C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\t...,C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\t...,None
1,lipo,lipo,ct_LIPO,tabpfn,0.521739,0.518095,0.681818,C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\t...,C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\t...,C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\t...,None


In [4]:
# LoCalPFN
res_loc = run_multi_localpfn(
    config_path=config_path,
    dataset_names=datasets_to_run,
    outputs_base_dir=outputs_base,
    checkpoint=checkpoint_path,
    skip_existing_embeddings=False,
    local_k=8,
    local_fit_adapter=True,
    local_adapter_epochs=8,
    local_adapter_num_queries=150,  # optional
)
res_loc['summary_df']



✅ gist: Reusing existing embeddings...
Prepared 25 cases ...
Prepared 50 cases ...
Prepared 75 cases ...
Prepared 100 cases ...
Prepared 125 cases ...
Prepared 150 cases ...
Prepared 175 cases ...
Prepared 200 cases ...
Prepared 225 cases ...
Done. Prepared 246 cases to C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\train\gist\ct_GIST
Validation set copied to: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\validation\gist\ct_GIST | Train: 196 | Val: 50
Loaded checkpoint (model_state_dict)
To extract: 246 from C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\train\gist\ct_GIST\imagesTr
Done extracting to: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\features\gist\ct_GIST_train
To extract: 50 from C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\validation\gist\ct_GIST\imagesVal
Done extracting to: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\features\gist\ct_G

c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


[adapter] epoch 1/8 loss=0.7797 a=0.993 b=0.002
[adapter] epoch 2/8 loss=0.7785 a=0.987 b=0.004
[adapter] epoch 3/8 loss=0.7774 a=0.980 b=0.006
[adapter] epoch 4/8 loss=0.7762 a=0.974 b=0.009
[adapter] epoch 5/8 loss=0.7751 a=0.967 b=0.011
[adapter] epoch 6/8 loss=0.7740 a=0.961 b=0.013
[adapter] epoch 7/8 loss=0.7729 a=0.955 b=0.015
[adapter] epoch 8/8 loss=0.7718 a=0.948 b=0.017
[LoCalPFN] Running local-context inference on validation set...

✅ lipo: Reusing existing embeddings...
Prepared 25 cases ...
Prepared 50 cases ...
Prepared 75 cases ...
Prepared 100 cases ...
Done. Prepared 115 cases to C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\train\lipo\ct_LIPO
Validation set copied to: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\validation\lipo\ct_LIPO | Train: 92 | Val: 23
Loaded checkpoint (model_state_dict)
To extract: 115 from C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\train\lipo\ct_LIPO\imagesTr
Done ext

,dataset,category,ct_name,method,accuracy,macro_f1,roc_auc,out_dir,metrics_path,pred_path,error
0,gist,gist,ct_GIST,localpfn,0.660000,0.659864,0.689600,C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\t...,C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\t...,C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\t...,None
1,lipo,lipo,ct_LIPO,localpfn,0.565217,0.557692,0.621212,C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\t...,C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\t...,C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\t...,None


## Run 3D baselines per dataset
We use the prepared SAM-Med3D folders and evaluate on the folder-level validation set (imagesVal).


In [5]:
from typing import Optional, Dict, Any, List

def _resolve_dataset_root(dataset_root: Optional[str], category: str, project_root: Path) -> Path:
    if dataset_root is not None:
        p = Path(dataset_root)
        if p.is_absolute() and p.exists():
            return p
        cand = (project_root / p).resolve()
        if cand.exists():
            return cand
    c1 = (project_root / category).resolve()
    if c1.exists():
        return c1
    c2 = (project_root / 'data' / category).resolve()
    if c2.exists():
        return c2
    raise FileNotFoundError(f'Could not resolve dataset_root for {category!r}. Tried: {dataset_root!r}, {c1}, {c2}')

sam3d_root = find_default_sam3d_root()
project_root = sam3d_root.parent.parent.resolve()

rows: List[Dict[str, Any]] = []
for ds_key in datasets_to_run:
    ds_cfg = cfg['datasets'][ds_key] or {}
    category = ds_cfg.get('category', ds_key)
    ct_name = ds_cfg.get('ct_name', f'ct_{ds_key.upper()}')
    ds_root = _resolve_dataset_root(ds_cfg.get('dataset_root'), category=category, project_root=project_root)
    labels = ds_cfg.get('labels', {}) or {}
    sheet_csv = labels.get('sheet_csv')
    sheet_path = (ds_root / sheet_csv) if sheet_csv else (ds_root / 'sheet.csv')
    dataset_name = labels.get('dataset_name')
    subject_col = labels.get('subject_col', 'Subject')
    label_col = labels.get('label_col', 'Diagnosis_binary')
    case_suffix = labels.get('case_suffix', '_CT')

    paths = Sam3DPaths(sam3d_root=sam3d_root, category=category, ct_name=ct_name)
    paths.ensure()

    # Keep 3D training manageable; adjust as needed
    d121 = None
    vit = None
    try:
        d121 = train_eval_densenet121_3d(
            paths=paths,
            sheet_csv=sheet_path,
            dataset_name=dataset_name,
            subject_col=subject_col,
            label_col=label_col,
            case_suffix=case_suffix,
            dataset_root=ds_root,
            epochs=4,  # adjust for speed/quality trade-off
            device=None,
        )
        er = d121['eval']
        rows.append({
            'dataset': ds_key, 'category': category, 'ct_name': ct_name, 'method': 'densenet121_3d',
            'accuracy': er['acc'], 'macro_f1': er['macro_f1'], 'roc_auc': er.get('roc_auc'),
            'out_dir': str(d121.get('out_dir', ''))
        })
    except Exception as e:
        rows.append({
            'dataset': ds_key, 'category': category, 'ct_name': ct_name, 'method': 'densenet121_3d',
            'accuracy': None, 'macro_f1': None, 'roc_auc': None, 'error': str(e)
        })
    try:
        vit = train_eval_vit_3d(
            paths=paths,
            sheet_csv=sheet_path,
            dataset_name=dataset_name,
            subject_col=subject_col,
            label_col=label_col,
            case_suffix=case_suffix,
            dataset_root=ds_root,
            epochs=4,  # adjust for speed/quality trade-off
            device=None,
        )
        er = vit['eval']
        rows.append({
            'dataset': ds_key, 'category': category, 'ct_name': ct_name, 'method': 'vit3d',
            'accuracy': er['acc'], 'macro_f1': er['macro_f1'], 'roc_auc': er.get('roc_auc'),
            'out_dir': str(vit.get('out_dir', ''))
        })
    except Exception as e:
        rows.append({
            'dataset': ds_key, 'category': category, 'ct_name': ct_name, 'method': 'vit3d',
            'accuracy': None, 'macro_f1': None, 'roc_auc': None, 'error': str(e)
        })

df_baselines = pd.DataFrame(rows)
df_baselines


c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\torchio\transforms\transform.py:158: RuntimeWarning: Output shape (96, 96, 24) != target shape (np.int64(96), np.int64(96), np.int64(25)). Fixing with CropOrPad
  transformed = self.apply_transform(subject)
c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\torchio\transforms\transform.py:158: RuntimeWarning: Output shape (96, 96, 13) != target shape (np.int64(96), np.int64(96), np.int64(14)). Fixing with CropOrPad
  transformed = self.apply_transform(subject)
c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\torchio\transforms\transform.py:158: RuntimeWarning: Output shape (96, 96, 14) != target shape (np.int64(96), np.int64(96), np.int64(15)). Fixing with CropOrPad
  transformed = self.apply_transform(subject)
c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\torchio\transforms\transform.py:158: RuntimeWarning: Output shape (96, 96, 29) != target shape (np.int64(96), np.int64(96), np.int64(30)). Fixing with CropOrPad
  

KeyboardInterrupt: 

## Combined summary


In [ ]:
summary_pfn = pd.concat([res_tab['summary_df'], res_loc['summary_df']], ignore_index=True)
combined = pd.concat([summary_pfn, df_baselines], ignore_index=True, sort=False)
combined


,dataset,category,ct_name,method,accuracy,macro_f1,roc_auc,out_dir,metrics_path,pred_path,error
0,gist,gist,ct_GIST,tabpfn,0.660000,0.659864,0.744000,C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\t...,C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\t...,C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\t...,None
1,lipo,lipo,ct_LIPO,tabpfn,0.521739,0.518095,0.515152,C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\t...,C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\t...,C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\t...,None
2,gist,gist,ct_GIST,localpfn,0.660000,0.659864,0.689600,C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\t...,C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\t...,C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\t...,None
3,lipo,lipo,ct_LIPO,localpfn,NaN,NaN,NaN,None,None,None,all input arrays must have the same shape\nTra...
4,gist,gist,ct_GIST,densenet121_3d,0.540000,0.386667,0.514610,C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\b...,NaN,NaN,NaN
5,gist,gist,ct_GIST,vit3d,0.480000,0.324324,0.362013,C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\b...,NaN,NaN,NaN
6,lipo,lipo,ct_LIPO,densenet121_3d,0.478261,0.469231,0.600000,C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\b...,NaN,NaN,NaN
7,lipo,lipo,ct_LIPO,vit3d,0.608696,0.608696,0.716667,C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\b...,NaN,NaN,NaN


In [ ]:
# Save combined summary to CSV
out_csv = outputs_base / 'combined_benchmarks_summary.csv'
combined.to_csv(out_csv, index=False)
print('Wrote:', out_csv)


Wrote: C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\combined_benchmarks_summary.csv
